# Step 1 — Data preparation with explicit sample-flow QC

This cleaned version preserves the original scientific preprocessing logic while making the analytic sample definition explicit and auditable.

Key changes relative to the older notebook:
- uses a single `ANALYSIS_TYPE` switch instead of three booleans;
- treats the matched ABCD imaging sample (`matched_group` 1/2) as the analysis backbone;
- uses **left / one-to-one merges** onto that backbone instead of outer merges;
- checks and prevents subject-level row multiplication during merges;
- replaces unrestricted `dropna()` across thousands of columns with an explicit complete-case requirement containing only variables needed for harmonization/downstream analysis;
- records exactly which subjects are excluded and why;
- writes sample-flow and exclusion-QC CSVs alongside the cleaned analysis file.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# -------------------------
# Configuration
# -------------------------
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
INPUT_DATA_DIR = ROOT_DIR / "input_data"
OUTPUT_DIR = ROOT_DIR / "output_data" / "cleaned"
QC_DIR = OUTPUT_DIR / "qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path(
    "/mnt/isilon/bgdlab_hbcd/projects/meisler_abcd_dmri/data/raw_data/merged_data_meisler_analyses.parquet"
)
ADI_PATH = Path(
    "/mnt/isilon/bgdlab_hbcd/shared_data/ABCD/LASSO_tabular/dairc/rawdata/phenotype/le_l_adi.parquet"
)

# Choose exactly one: "main", "cognition", "income"
ANALYSIS_TYPE = "main"
assert ANALYSIS_TYPE in {"main", "cognition", "income"}

BASELINE_SESSION = "ses-00A"
BASELINE_EVENT = "baseline_year_1_arm_1"
PARTICIPANTS_BASELINE_EVENT = "ses-baselineYear1Arm1"
ID_COL = "subject_id_clean"
GROUP_COL = "matched_group"

EXPECTED_MATCHED_COUNTS = {1.0: 4082, 2.0: 4101}
EXPECTED_MATCHED_TOTAL = sum(EXPECTED_MATCHED_COUNTS.values())

print("Analysis type:", ANALYSIS_TYPE)


Analysis type: main


In [3]:
# -------------------------
# Reusable QC helpers
# -------------------------
def report_sample(df, label, id_col=ID_COL, group_col=GROUP_COL):
    row = {
        "stage": label,
        "rows": len(df),
        "unique_subjects": df[id_col].nunique(dropna=True) if id_col in df.columns else np.nan,
    }
    if group_col in df.columns:
        counts = df[group_col].value_counts(dropna=False)
        row["group1_n"] = int(counts.get(1.0, 0))
        row["group2_n"] = int(counts.get(2.0, 0))
    else:
        row["group1_n"] = np.nan
        row["group2_n"] = np.nan

    print(f"\n[{label}]")
    print("rows:", row["rows"], "| unique subjects:", row["unique_subjects"])
    if group_col in df.columns:
        print(df[group_col].value_counts(dropna=False).sort_index())
    return row


def require_unique_subjects(df, name, id_col=ID_COL, allow_exact_duplicates=True):
    """Return one row per subject; never silently resolve conflicting duplicate rows."""
    if id_col not in df.columns:
        raise KeyError(f"{name}: missing ID column {id_col!r}")

    dup_mask = df[id_col].duplicated(keep=False)
    if not dup_mask.any():
        return df.copy()

    dup = df.loc[dup_mask].copy()
    dup_path = QC_DIR / f"{ANALYSIS_TYPE}_{name}_duplicate_subject_rows.csv"
    dup.to_csv(dup_path, index=False)

    if not allow_exact_duplicates:
        raise ValueError(
            f"{name}: duplicate subject IDs found. Rows saved to {dup_path}."
        )

    # Exact duplicates across all supplied columns can be safely collapsed.
    conflicting_ids = []
    for sid, g in dup.groupby(id_col, dropna=False):
        if len(g.drop_duplicates()) > 1:
            conflicting_ids.append(sid)

    if conflicting_ids:
        raise ValueError(
            f"{name}: {len(conflicting_ids)} subject IDs have non-identical duplicate rows. "
            f"Resolve them before merging. Details: {dup_path}"
        )

    print(
        f"[QC] {name}: collapsing {len(dup) - dup[id_col].nunique()} exact duplicate rows "
        f"across {dup[id_col].nunique()} duplicated subjects."
    )
    return df.drop_duplicates().copy()


def clean_coded_column(df, col, valid_range, missing_codes=(777, 999)):
    out = df.copy()
    s = pd.to_numeric(out[col], errors="coerce")
    s = s.replace({code: np.nan for code in missing_codes})
    lo, hi = valid_range
    out[col] = s.where((s >= lo) & (s <= hi), np.nan)
    return out


def missing_fields_by_subject(df, required_cols, id_col=ID_COL):
    miss = df[required_cols].isna()
    bad = miss.any(axis=1)
    if not bad.any():
        return pd.DataFrame(columns=[id_col, "missing_fields", "n_missing_required"])
    out = df.loc[bad, [id_col]].copy()
    out["missing_fields"] = miss.loc[bad].apply(
        lambda r: ";".join(r.index[r].tolist()), axis=1
    )
    out["n_missing_required"] = miss.loc[bad].sum(axis=1).astype(int).to_numpy()
    return out


## 1. Load baseline imaging data and construct the matched ABCD backbone


In [4]:
data = pd.read_parquet(DATA_PATH, engine="fastparquet")
data = data.loc[data["session_id"] == BASELINE_SESSION].copy()
print("Baseline imaging rows:", data.shape)


def filter_columns_by_tracts_and_metrics(columns):
    selected = [
        "subject_session", "subject_id", "session_id", "age", "site", "sex",
        "t1post_dwi_contrast", "batch_device_software", "eTIV",
        "scanner_model", "scanner_manufacturer",
    ]
    selected += [c for c in columns if c.startswith("bundle_")]
    # preserve order while removing accidental duplicates
    return list(dict.fromkeys(selected))

selected_columns = filter_columns_by_tracts_and_metrics(data.columns)
data_reduced = data[selected_columns].copy()
data_reduced[ID_COL] = data_reduced["subject_id"].astype(str).str.replace(r"^sub-", "", regex=True)

# One baseline imaging row per subject is required for all downstream merges.
data_reduced = require_unique_subjects(data_reduced, "baseline_imaging", allow_exact_duplicates=False)
print("Reduced baseline imaging shape:", data_reduced.shape)


Baseline imaging rows: (8402, 5499)
Reduced baseline imaging shape: (8402, 5406)


In [5]:
participants = pd.read_csv(INPUT_DATA_DIR / "participants.tsv", sep="\t")
participants = participants.loc[
    participants["session_id"] == PARTICIPANTS_BASELINE_EVENT
].copy()
participants[ID_COL] = participants["participant_id"].astype(str).str.replace(
    r"^sub-NDARINV", "", regex=True
)

# Participant table should contribute a unique matched-group assignment per subject.
matched_lookup = participants[[ID_COL, GROUP_COL]].drop_duplicates()
conflicting_group_ids = (
    matched_lookup.groupby(ID_COL)[GROUP_COL].nunique(dropna=False).loc[lambda s: s > 1]
)
if len(conflicting_group_ids):
    raise ValueError(
        f"Found {len(conflicting_group_ids)} subjects with conflicting matched_group values."
    )
matched_lookup = matched_lookup.drop_duplicates(subset=[ID_COL]).set_index(ID_COL)[GROUP_COL]

data_reduced[GROUP_COL] = data_reduced[ID_COL].map(matched_lookup)

print("Matched-group counts before filtering:")
print(data_reduced[GROUP_COL].value_counts(dropna=False))

# Keep only the two prespecified ARMS matched split halves.
data_reduced = data_reduced.loc[data_reduced[GROUP_COL].isin([1.0, 2.0])].copy()

flow = [report_sample(data_reduced, "01_matched_ABCD_backbone")]

observed_counts = data_reduced[GROUP_COL].value_counts().to_dict()
for grp, expected in EXPECTED_MATCHED_COUNTS.items():
    observed = int(observed_counts.get(grp, 0))
    if observed != expected:
        raise ValueError(
            f"Matched group {grp:g}: observed {observed}, expected {expected}. "
            "Resolve this before proceeding."
        )
assert data_reduced[ID_COL].nunique() == EXPECTED_MATCHED_TOTAL


Matched-group counts before filtering:
matched_group
2.0      4101
1.0      4082
3.0       216
NaN         2
888.0       1
Name: count, dtype: int64

[01_matched_ABCD_backbone]
rows: 8183 | unique subjects: 8183
matched_group
1.0    4082
2.0    4101
Name: count, dtype: int64


## 2. Load exposure / cognition / income variables

All supplemental tables are reduced to **one baseline row per subject before merging**. The matched imaging cohort remains the backbone; supplemental data are merged with `how="left"` and `validate="one_to_one"` so extra participants cannot inflate the row count.


In [9]:
# ============================================================
# Exposome factor scores
# ============================================================

exposome = pd.read_csv(
    INPUT_DATA_DIR / "abcd_longitudinal_factor_scores_7f_bifactor.csv"
)

exposome[ID_COL] = (
    exposome["ID"]
    .astype(str)
    .str.replace(r"^NDAR_INV", "", regex=True)
)

exposome_filtered = exposome.loc[
    exposome["Timepoint"] == BASELINE_EVENT,
    [
        ID_COL,
        "General_SES",
        "School",
        "Family_Values",
        "Family_Turmoil",
        "Dense_Urban_Poverty",
        "Extracurriculars",
        "Screen_Time",
    ],
].copy()

exposome_filtered = require_unique_subjects(
    exposome_filtered,
    "baseline_exposome"
)

print(
    "Baseline exposome unique subjects:",
    exposome_filtered[ID_COL].nunique()
)


# ============================================================
# Cognition scores
# ============================================================

cognition_data = pd.read_csv(
    INPUT_DATA_DIR / "cognition_data.csv"
)

cognition_core = cognition_data.loc[
    cognition_data["event_name"] == BASELINE_EVENT,
    [
        "src_subject_id",
        "neurocog_pc1.bl",
        "neurocog_pc2.bl",
        "neurocog_pc3.bl",
    ],
].copy()

cognition_core[ID_COL] = (
    cognition_core["src_subject_id"]
    .astype(str)
    .str.replace(r"^NDAR_INV", "", regex=True)
)

cognition_core = cognition_core.drop(
    columns=["src_subject_id"]
)

cognition_core = require_unique_subjects(
    cognition_core,
    "baseline_cognition"
)

print(
    "Baseline cognition unique subjects:",
    cognition_core[ID_COL].nunique()
)


# ============================================================
# ADI / income sensitivity variables
# ============================================================

adi = pd.read_parquet(ADI_PATH)

adi = adi.loc[
    adi["session_id"] == BASELINE_SESSION
].copy()

adi[ID_COL] = (
    adi["participant_id"]
    .astype(str)
    .str.replace(r"^sub-", "", regex=True)
)

adi_lookup = (
    adi[
        [
            ID_COL,
            "le_l_adi__addr1__national_prcnt",
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Check whether any subject has >1 distinct ADI value
adi_conflicts = (
    adi_lookup
    .groupby(ID_COL)["le_l_adi__addr1__national_prcnt"]
    .nunique(dropna=False)
)

adi_conflicting_ids = adi_conflicts.loc[
    adi_conflicts > 1
].index

if len(adi_conflicting_ids) > 0:
    display(
        adi_lookup.loc[
            adi_lookup[ID_COL].isin(adi_conflicting_ids)
        ].sort_values(ID_COL)
    )
    raise ValueError(
        f"{len(adi_conflicting_ids)} subjects have conflicting "
        "baseline ADI values."
    )

# Now safe to collapse to one row per subject
adi_lookup = (
    adi_lookup
    .drop_duplicates(subset=[ID_COL])
    .copy()
)

assert adi_lookup[ID_COL].is_unique

print(
    "Baseline ADI unique subjects:",
    adi_lookup[ID_COL].nunique()
)


# ============================================================
# Parent education + household income lookup
# ============================================================

# Only carry the variables needed for this sensitivity analysis.
participants_sens = participants[
    [
        ID_COL,
        "parental_education",
        "income",
    ]
].copy()

participants_sens = clean_coded_column(
    participants_sens,
    "parental_education",
    valid_range=(0, 21),
    missing_codes=(777, 999),
)

participants_sens = clean_coded_column(
    participants_sens,
    "income",
    valid_range=(1, 9),
    missing_codes=(777, 999),
)

# ------------------------------------------------------------
# QC repeated participant rows BEFORE deduplicating
# ------------------------------------------------------------

dup_mask = participants_sens[ID_COL].duplicated(
    keep=False
)

n_dup_rows = int(dup_mask.sum())
n_dup_subjects = int(
    participants_sens.loc[
        dup_mask,
        ID_COL
    ].nunique()
)

print(
    f"Participant SES lookup before deduplication: "
    f"{len(participants_sens)} rows, "
    f"{participants_sens[ID_COL].nunique()} unique subjects"
)

print(
    f"Repeated SES records: "
    f"{n_dup_rows} rows across "
    f"{n_dup_subjects} subjects"
)

# Check that duplicated subjects do not disagree on SES values
ses_check = (
    participants_sens
    .groupby(ID_COL)[
        [
            "parental_education",
            "income",
        ]
    ]
    .nunique(dropna=False)
)

ses_conflicting_ids = ses_check.loc[
    (ses_check["parental_education"] > 1)
    | (ses_check["income"] > 1)
].index

if len(ses_conflicting_ids) > 0:
    display(
        participants_sens.loc[
            participants_sens[ID_COL].isin(
                ses_conflicting_ids
            )
        ].sort_values(ID_COL)
    )

    raise ValueError(
        f"{len(ses_conflicting_ids)} subjects have conflicting "
        "parental education and/or income values across repeated "
        "baseline rows. Resolve before deduplicating."
    )

print(
    "Subjects with conflicting SES values:",
    len(ses_conflicting_ids)
)

# Safe because we verified duplicates do not disagree
participants_sens = (
    participants_sens
    .drop_duplicates(subset=[ID_COL])
    .copy()
)

assert participants_sens[ID_COL].is_unique

print(
    "Participant SES lookup after deduplication:",
    len(participants_sens),
    "unique subjects"
)


# ============================================================
# Merge ADI onto SES lookup
# ============================================================

participants_sens = participants_sens.merge(
    adi_lookup,
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

assert participants_sens[ID_COL].is_unique


# ============================================================
# Final income / education sensitivity lookup
# ============================================================

income_core = participants_sens[
    [
        ID_COL,
        "parental_education",
        "income",
        "le_l_adi__addr1__national_prcnt",
    ]
].copy()

assert income_core[ID_COL].is_unique

print(
    "Income/education/ADI sensitivity lookup subjects:",
    income_core[ID_COL].nunique()
)

print("\nMissingness in sensitivity variables:")
display(
    income_core[
        [
            "parental_education",
            "income",
            "le_l_adi__addr1__national_prcnt",
        ]
    ]
    .isna()
    .sum()
    .rename("n_missing")
    .to_frame()
)

Baseline exposome unique subjects: 11874
Baseline cognition unique subjects: 11878
Baseline ADI unique subjects: 11214
Participant SES lookup before deduplication: 11930 rows, 11886 unique subjects
Repeated SES records: 88 rows across 44 subjects
Subjects with conflicting SES values: 0
Participant SES lookup after deduplication: 11886 unique subjects
Income/education/ADI sensitivity lookup subjects: 11886

Missingness in sensitivity variables:


,n_missing
parental_education,31
income,2278
le_l_adi__addr1__national_prcnt,898


## 3. Merge onto the matched imaging backbone and audit missing supplemental records


In [10]:
merged = data_reduced.copy()

if ANALYSIS_TYPE in {"main", "cognition"}:
    matched_ids = set(merged[ID_COL])
    exposome_ids = set(exposome_filtered[ID_COL])
    no_exposome = sorted(matched_ids - exposome_ids)
    print("Matched participants without baseline exposome row:", len(no_exposome))
    pd.DataFrame({ID_COL: no_exposome}).to_csv(
        QC_DIR / f"{ANALYSIS_TYPE}_matched_subjects_without_exposome.csv", index=False
    )

    merged = merged.merge(
        exposome_filtered,
        on=ID_COL,
        how="left",
        validate="one_to_one",
        indicator="_exposome_merge",
    )
    print(merged["_exposome_merge"].value_counts(dropna=False))
    merged = merged.drop(columns="_exposome_merge")
    flow.append(report_sample(merged, "02_after_exposome_left_merge"))

if ANALYSIS_TYPE == "cognition":
    matched_ids = set(merged[ID_COL])
    cog_ids = set(cognition_core[ID_COL])
    no_cognition = sorted(matched_ids - cog_ids)
    print("Matched participants without baseline cognition row:", len(no_cognition))
    pd.DataFrame({ID_COL: no_cognition}).to_csv(
        QC_DIR / "cognition_matched_subjects_without_cognition.csv", index=False
    )

    merged = merged.merge(
        cognition_core,
        on=ID_COL,
        how="left",
        validate="one_to_one",
        indicator="_cognition_merge",
    )
    print(merged["_cognition_merge"].value_counts(dropna=False))
    merged = merged.drop(columns="_cognition_merge")
    flow.append(report_sample(merged, "03_after_cognition_left_merge"))

if ANALYSIS_TYPE == "income":
    merged = merged.merge(
        income_core,
        on=ID_COL,
        how="left",
        validate="one_to_one",
        indicator="_income_merge",
    )
    print(merged["_income_merge"].value_counts(dropna=False))
    merged = merged.drop(columns="_income_merge")
    flow.append(report_sample(merged, "02_after_income_left_merge"))

assert len(merged) == EXPECTED_MATCHED_TOTAL, "A left one-to-one merge should not change row count."
assert merged[ID_COL].nunique() == EXPECTED_MATCHED_TOTAL


Matched participants without baseline exposome row: 2
_exposome_merge
both          8181
left_only        2
right_only       0
Name: count, dtype: int64

[02_after_exposome_left_merge]
rows: 8183 | unique subjects: 8183
matched_group
1.0    4082
2.0    4101
Name: count, dtype: int64


## 4. Remove unused WM variables and define the complete-case analysis requirements

The older notebook called `dropna()` across the entire ~5,000-column dataframe. Here, exclusion is based only on variables actually required for the selected analysis:

- all retained WM features;
- harmonization batch;
- age / sex;
- downstream image-quality covariate (`t1post_dwi_contrast`);
- analysis-specific environmental/cognitive covariates.

This makes every exclusion interpretable and prevents unrelated metadata missingness from defining the sample.


In [11]:
bundles_to_remove = [
    "bundle_ProjectionBrainstem_DentatorubrothalamicTractL",
    "bundle_Projection_BrainstemDentatorubrothalamicTractR",
    "bundle_Commissure_AnteriorCommissure",
    "bundle_ProjectionBrainstem_CorticobulbarTractL",
    "bundle_ProjectionBrainstem_CorticobulbarTractR",
]
metrics_to_remove = ["number_of_tracts", "NODDI_directions"]

cols_to_drop = [
    c for c in merged.columns if any(bundle in c for bundle in bundles_to_remove)
]
merged = merged.drop(columns=cols_to_drop)
print("Dropped bundle columns:", len(cols_to_drop))

metric_cols_to_drop = [
    c for c in merged.columns if any(metric in c for metric in metrics_to_remove)
]
merged = merged.drop(columns=metric_cols_to_drop)
print("Dropped metric columns:", len(metric_cols_to_drop))

wm_cols = [c for c in merged.columns if c.startswith("bundle_")]
print("Retained WM columns:", len(wm_cols))

BASE_REQUIRED = [
    ID_COL, GROUP_COL, "age", "sex", "t1post_dwi_contrast", "batch_device_software"
]

if ANALYSIS_TYPE == "main":
    ANALYSIS_REQUIRED = [
        "General_SES", "School", "Family_Values", "Family_Turmoil",
        "Dense_Urban_Poverty", "Extracurriculars", "Screen_Time",
    ]
elif ANALYSIS_TYPE == "cognition":
    ANALYSIS_REQUIRED = [
        "General_SES", "School", "Family_Values", "Family_Turmoil",
        "Dense_Urban_Poverty", "Extracurriculars", "Screen_Time",
        "neurocog_pc1.bl", "neurocog_pc2.bl", "neurocog_pc3.bl",
    ]
else:
    ANALYSIS_REQUIRED = [
        "parental_education", "income", "le_l_adi__addr1__national_prcnt"
    ]

required_cols = list(dict.fromkeys(BASE_REQUIRED + ANALYSIS_REQUIRED + wm_cols))
missing_required_columns = [c for c in required_cols if c not in merged.columns]
if missing_required_columns:
    raise KeyError(f"Required columns missing from merged dataframe: {missing_required_columns[:20]}")

missing_qc = missing_fields_by_subject(merged, required_cols)
missing_qc.to_csv(
    QC_DIR / f"{ANALYSIS_TYPE}_subjects_removed_required_complete_case.csv",
    index=False,
)
print("Subjects failing explicit complete-case requirements:", len(missing_qc))
if len(missing_qc):
    display(missing_qc.head(20))

before_ids = set(merged[ID_COL])
cleaned = merged.dropna(subset=required_cols).copy()
after_ids = set(cleaned[ID_COL])
removed_ids = sorted(before_ids - after_ids)

print("Rows before explicit complete-case filter:", len(merged))
print("Rows after explicit complete-case filter:", len(cleaned))
print("Subjects removed:", len(removed_ids))
flow.append(report_sample(cleaned, "04_after_required_complete_case_filter"))

# Downstream assumptions
assert not cleaned[required_cols].isna().any().any()
assert not cleaned[ID_COL].duplicated().any()


Dropped bundle columns: 0
Dropped metric columns: 62
Retained WM columns: 5332
Subjects failing explicit complete-case requirements: 2


,subject_id_clean,missing_fields,n_missing_required
670,2HYAENE6,General_SES;School;Family_Values;Family_Turmoi...,7
3070,C68NXGHP,General_SES;School;Family_Values;Family_Turmoi...,7


Rows before explicit complete-case filter: 8183
Rows after explicit complete-case filter: 8181
Subjects removed: 2

[04_after_required_complete_case_filter]
rows: 8181 | unique subjects: 8181
matched_group
1.0    4081
2.0    4100
Name: count, dtype: int64


## 5. Save cleaned data and QC tables


In [12]:
OUTPUT_FILES = {
    "main": "exposome_FINAL_clean_qc.csv",
    "cognition": "exposome_FINAL_cognition_clean_qc.csv",
    "income": "parental_edu_income_sensitivity_clean_qc.csv",
}
output_file = OUTPUT_DIR / OUTPUT_FILES[ANALYSIS_TYPE]
cleaned.to_csv(output_file, index=False)

flow_df = pd.DataFrame(flow)
flow_file = QC_DIR / f"{ANALYSIS_TYPE}_step1_sample_flow.csv"
flow_df.to_csv(flow_file, index=False)

print("\nFinal Step 1 sample flow:")
display(flow_df)
print("\nSaved cleaned data:", output_file)
print("Saved sample flow:", flow_file)
print("QC directory:", QC_DIR)


Final Step 1 sample flow:


,stage,rows,unique_subjects,group1_n,group2_n
0,01_matched_ABCD_backbone,8183,8183,4082,4101
1,02_after_exposome_left_merge,8183,8183,4082,4101
2,04_after_required_complete_case_filter,8181,8181,4081,4100



Saved cleaned data: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/exposome_FINAL_clean_qc.csv
Saved sample flow: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/qc/main_step1_sample_flow.csv
QC directory: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/qc


### Expected interpretation for the main analysis

If this reproduces the historical pipeline, the likely flow is:

- 8,183 matched ABCD participants (4,082 / 4,101)
- 8,175 after the explicit Step-1 required-data filter
- Step 2 then removes any globally singleton CovBat batch, yielding 8,174 total

The QC CSVs above will identify the exact subject IDs and missing fields rather than inferring the losses from row counts.
